# Notebook 09: Out-of-Sample Backtest — 10-Year e₀ Hindcast

## Purpose
The strongest possible validation: train the AINN on data **up to 2010 only**, forecast mortality recursively for 2011-2020, and compare predicted life expectancy with the actual HMD-observed values.

This tests the model's real-world predictive capability — not one-step-ahead RMSE on scaled differences, but the ability to project life expectancy over a decade that includes the post-2011 mortality deceleration and the 2020 COVID shock.

## Design
1. **Data truncation**: Use only years 1956-2010 (54 first differences).
2. **Li-Lee re-extraction**: Recompute common and specific factors on the truncated data.
3. **AINN training**: Train champion architecture (LSTM 48-32, lb=15) on 1956-2010.
4. **Recursive forecast**: 10-year MC Dropout forecast (2011-2020), 200 simulations.
5. **e₀ reconstruction**: Observation-anchored at 2010 with isotonic Gompertz.
6. **Ground truth**: Compute observed e₀ from HMD mortality for 2011-2020.
7. **Li-Lee RWD baseline**: Same backtest with Random Walk with Drift for comparison.
8. **Evaluation**: Predicted vs observed e₀ trajectories.


## 9.1: Setup

In [ ]:
import sys
sys.path.append('../src')
from reproducibility import set_seed
set_seed(42)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pickle, os, logging, warnings, time
warnings.filterwarnings('ignore')

os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
import tensorflow as tf
tf.get_logger().setLevel(logging.ERROR)
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.preprocessing import StandardScaler
from sklearn.isotonic import IsotonicRegression

from style_config import set_style, save_dual, COUNTRIES, COUNTRY_COLORS
set_style('notebook')

PROCESSED_DIR = '../data/processed/'
FIGURES_DIR = '../reports/figures/'

# Load full Li-Lee parameters (1956-2020)
with open(os.path.join(PROCESSED_DIR, 'li_lee_params.pkl'), 'rb') as f:
    bundle = pickle.load(f)

YEARS = bundle['metadata']['years']
AGES = bundle['metadata']['ages']
N_AGES = len(AGES)

# Champion config
with open(os.path.join(PROCESSED_DIR, 'training_meta.pkl'), 'rb') as f:
    meta = pickle.load(f)

LOOKBACK = meta['lookback']
U1, U2 = meta['units_l1'], meta['units_l2']
LR = meta['lr']
LC = meta['lambda_coherence']
LM = meta['lambda_monotonicity']

# Backtest parameters
CUTOFF_YEAR = 2010
CUTOFF_IDX = list(YEARS).index(CUTOFF_YEAR)  # 54
N_BACKTEST = 10  # 2011-2020
N_SIMS = 200
BATCH_SIZE = 8
EPOCHS = 150

print(f'Champion config: LSTM({U1},{U2}), lb={LOOKBACK}, lr={LR}')
print(f'Backtest: train on 1956-{CUTOFF_YEAR}, forecast {CUTOFF_YEAR+1}-{YEARS[-1]}')
print(f'Cutoff index: {CUTOFF_IDX} (of {len(YEARS)} years)')

## 9.2: Compute Observed e₀ (Ground Truth)

Calculate the actual life expectancy from HMD mortality data for every year 2010-2020.

In [ ]:
_IR = IsotonicRegression(increasing=True)
_AGES_40_90 = np.arange(40, 91)

def calculate_e0(log_mx):
    mx = np.exp(log_mx)
    qx = 1.0 - np.exp(-mx)
    qx[-1] = 1.0
    px = 1.0 - qx
    lx = np.concatenate(([1.0], np.cumprod(px[:-1])))
    lx_ext = np.append(lx, 0.0)
    return np.sum((lx_ext[:-1] + lx_ext[1:]) / 2.0)

# Compute observed e0 for all years, all countries, both sexes
observed_e0 = {}  # {sex: {code: array of e0 for years[CUTOFF_IDX:]}}
backtest_years = YEARS[CUTOFF_IDX:]  # 2010, 2011, ..., 2020 (11 years)

for sex in ['male', 'female']:
    observed_e0[sex] = {}
    for code in COUNTRIES.keys():
        log_mx_matrix = np.load(os.path.join(PROCESSED_DIR, f'{code}_log_mx_{sex}.npy'))
        e0_series = []
        for yr_idx in range(CUTOFF_IDX, len(YEARS)):
            log_mx_yr = log_mx_matrix[:, yr_idx].copy()
            log_mx_yr[40:91] = _IR.fit_transform(_AGES_40_90, log_mx_yr[40:91])
            e0_series.append(calculate_e0(log_mx_yr))
        observed_e0[sex][code] = np.array(e0_series)

print(f'Observed e0 computed for years {backtest_years[0]}-{backtest_years[-1]}')
print(f'CHE Male 2010: {observed_e0["male"]["CHE"][0]:.2f}')
print(f'CHE Male 2020: {observed_e0["male"]["CHE"][-1]:.2f}')
print(f'CHE Female 2010: {observed_e0["female"]["CHE"][0]:.2f}')
print(f'CHE Female 2020: {observed_e0["female"]["CHE"][-1]:.2f}')

## 9.3: Prepare Truncated Data & Train Backtest Model

Re-extract Li-Lee factors on 1956-2010 data only, prepare training sequences, and train the AINN with the champion configuration.

In [ ]:
# Truncate feature matrices to 1956-2010
# feature_matrices are first differences: (64, 7) for full 1957-2020
# Truncate to 1957-2010: first CUTOFF_IDX-1 = 53 rows
# (because diff of 55 years = 54 diffs, indices 0-53)
n_diffs_truncated = CUTOFF_IDX - 1  # 53 diffs (1957-2010)

feature_matrices_trunc = {
    'male': bundle['feature_matrices']['male'][:n_diffs_truncated + 1],   # 0:54 → 54 rows (1957-2010)
    'female': bundle['feature_matrices']['female'][:n_diffs_truncated + 1]
}

# For the backtest, we use ALL truncated data for training (no val holdout)
# because the test set IS the 2011-2020 e0 comparison.
# But we still need a validation set for early stopping.
# Split: last 9 diffs as val (2002-2010), rest as train.
TRAIN_END = n_diffs_truncated + 1 - 9  # Train up to ~2001, val 2002-2010

class AINNLoss(keras.losses.Loss):
    def __init__(self, lambda_coherence=0.0, lambda_monotonicity=0.0, **kwargs):
        super().__init__(**kwargs)
        self.lambda_coherence = lambda_coherence
        self.lambda_monotonicity = lambda_monotonicity
    def call(self, y_true, y_pred):
        mse = tf.reduce_mean(tf.square(y_true - y_pred))
        coherence = tf.reduce_mean(tf.square(y_pred[:, 1:7]))
        monotonicity = tf.reduce_mean(tf.square(tf.nn.relu(y_pred[:, 0])))
        return mse + self.lambda_coherence * coherence + self.lambda_monotonicity * monotonicity

def prepare_backtest_data(lookback):
    """Prepare joint M/F sequences from truncated (1956-2010) data."""
    male_data = np.column_stack([feature_matrices_trunc['male'], 
                                  np.zeros(len(feature_matrices_trunc['male']))])
    female_data = np.column_stack([feature_matrices_trunc['female'],
                                    np.ones(len(feature_matrices_trunc['female']))])
    
    train_combined = np.vstack([male_data[:TRAIN_END], female_data[:TRAIN_END]])
    scaler = StandardScaler()
    scaler.fit(train_combined)
    
    male_scaled = scaler.transform(male_data)
    female_scaled = scaler.transform(female_data)
    
    def make_seq(data, lb):
        X, y = [], []
        for i in range(lb, len(data)):
            X.append(data[i-lb:i])
            y.append(data[i])
        return np.array(X), np.array(y)
    
    X_m, y_m = make_seq(male_scaled, lookback)
    X_f, y_f = make_seq(female_scaled, lookback)
    
    n_train = TRAIN_END - lookback
    X_train = np.concatenate([X_m[:n_train], X_f[:n_train]])
    y_train = np.concatenate([y_m[:n_train], y_f[:n_train]])
    X_val = np.concatenate([X_m[n_train:], X_f[n_train:]])
    y_val = np.concatenate([y_m[n_train:], y_f[n_train:]])
    
    idx = np.random.permutation(len(X_train))
    X_train, y_train = X_train[idx], y_train[idx]
    
    return X_train, y_train, X_val, y_val, scaler

# Prepare data and train
set_seed(42)
X_train, y_train, X_val, y_val, scaler_bt = prepare_backtest_data(LOOKBACK)

print(f'Truncated data: {n_diffs_truncated + 1} first differences (1957-2010)')
print(f'Training sequences: {X_train.shape[0]} (train) + {X_val.shape[0]} (val)')
print(f'Sequence shape: {X_train.shape[1:]} (lookback={LOOKBACK}, features=8)')

# Train backtest model
set_seed(42)
inputs = layers.Input(shape=(LOOKBACK, 8))
x = layers.LSTM(U1, return_sequences=True)(inputs)
x = layers.Dropout(0.2)(x)
x = layers.LSTM(U2)(x)
x = layers.Dropout(0.2)(x)
outputs = layers.Dense(8)(x)
bt_model = keras.Model(inputs=inputs, outputs=outputs)

bt_model.compile(optimizer=keras.optimizers.Adam(learning_rate=LR),
                 loss=AINNLoss(lambda_coherence=LC, lambda_monotonicity=LM))

es = EarlyStopping(monitor='val_loss', patience=20, restore_best_weights=True, verbose=0)
t0 = time.time()
hist = bt_model.fit(X_train, y_train, epochs=EPOCHS, batch_size=BATCH_SIZE,
                    validation_data=(X_val, y_val), callbacks=[es], verbose=0)
elapsed = time.time() - t0

print(f'\nBacktest model trained: {len(hist.history["loss"])} epochs, {elapsed:.0f}s')

## 9.4: Backtest Forecast (AINN + Li-Lee RWD)

Forecast 2011-2020 recursively from the 2010 anchor point, using both:
1. The AINN backtest model (MC Dropout, 200 sims)
2. Li-Lee RWD (1,000 sims, for comparison)

In [ ]:
# === AINN Forecast ===

def get_init_seq_backtest(sex):
    """Initial sequence: last LOOKBACK diffs of truncated data, scaled."""
    si = 0.0 if sex == 'male' else 1.0
    data = np.column_stack([feature_matrices_trunc[sex], 
                            np.full(len(feature_matrices_trunc[sex]), si)])
    return scaler_bt.transform(data)[-LOOKBACK:]

def forecast_mcd(model, init_seq, n_steps, n_sims=N_SIMS):
    n_feat = init_seq.shape[-1]
    sims = np.zeros((n_sims, n_steps, n_feat))
    for s in range(n_sims):
        seq = init_seq.copy()
        for t in range(n_steps):
            inp = tf.convert_to_tensor(seq[np.newaxis, ...], dtype=tf.float32)
            pred = model(inp, training=True).numpy().reshape(n_feat)
            sims[s, t, :] = pred
            seq = np.roll(seq, -1, axis=0)
            seq[-1] = pred
    return sims

# Compute residual σ on truncated data (walk-forward on 1956-2010)
def compute_residual_sigma_bt(sex):
    si = 0.0 if sex == 'male' else 1.0
    data = np.column_stack([feature_matrices_trunc[sex], 
                            np.full(len(feature_matrices_trunc[sex]), si)])
    data_scaled = scaler_bt.transform(data)
    residuals = []
    for t in range(LOOKBACK, len(data)):
        inp = tf.convert_to_tensor(data_scaled[t-LOOKBACK:t][np.newaxis, ...], dtype=tf.float32)
        pred = bt_model(inp, training=False).numpy().reshape(8)
        pred_orig = scaler_bt.inverse_transform(pred.reshape(1, -1)).reshape(8)
        residuals.append(data[t] - pred_orig)
    return np.std(np.array(residuals), axis=0)

print('Computing backtest residual σ...')
sigma_bt_m = compute_residual_sigma_bt('male')
sigma_bt_f = compute_residual_sigma_bt('female')
print(f'  Male Kt σ: {sigma_bt_m[0]:.4f}')
print(f'  Female Kt σ: {sigma_bt_f[0]:.4f}')

print('\nAINN backtest forecast (2011-2020)...')
init_m = get_init_seq_backtest('male')
init_f = get_init_seq_backtest('female')

sims_m_sc = forecast_mcd(bt_model, init_m, N_BACKTEST)
sims_f_sc = forecast_mcd(bt_model, init_f, N_BACKTEST)

# Inverse + residual noise
def inv_noise(sims_sc, sigma):
    n_s, n_t, n_f = sims_sc.shape
    flat = scaler_bt.inverse_transform(sims_sc.reshape(-1, n_f))
    orig = flat.reshape(n_s, n_t, n_f)
    noisy = orig.copy()
    for t in range(n_t):
        noisy[:, t, :] += np.random.normal(0, sigma, size=(n_s, n_f))
    return noisy

set_seed(42)
sims_m = inv_noise(sims_m_sc, sigma_bt_m)
sims_f = inv_noise(sims_f_sc, sigma_bt_f)

# === Li-Lee RWD Forecast ===
print('Li-Lee RWD backtest forecast (2011-2020)...')

def lilee_rwd_backtest(sex, n_sims=1000):
    Kt = bundle['common_factors'][sex]['Kt'][:CUTOFF_IDX + 1]  # Up to 2010
    dKt = np.diff(Kt)
    mu, sigma = dKt.mean(), dKt.std()
    delta_sims = mu + sigma * np.random.randn(n_sims, N_BACKTEST)
    return delta_sims, mu, sigma

set_seed(42)
ll_delta_m, ll_mu_m, ll_sig_m = lilee_rwd_backtest('male')
ll_delta_f, ll_mu_f, ll_sig_f = lilee_rwd_backtest('female')

print(f'Li-Lee RWD params (truncated to 2010):')
print(f'  Male: μ={ll_mu_m:.4f}, σ={ll_sig_m:.4f}')
print(f'  Female: μ={ll_mu_f:.4f}, σ={ll_sig_f:.4f}')
print(f'\nForecasts ready: AINN {sims_m.shape}, Li-Lee {ll_delta_m.shape}')

## 9.5: Reconstruct e₀ and Compare with Observed

In [ ]:
def reconstruct_e0_backtest(delta_sims, sex, anchor_year_idx=CUTOFF_IDX):
    """Observation-anchored e0 reconstruction from first-difference simulations."""
    Bx = bundle['common_factors'][sex]['Bx']
    n_sims, n_steps = delta_sims.shape[0], delta_sims.shape[1]
    n_countries = len(COUNTRIES)
    
    # Use only column 0 (common factor Kt)
    if delta_sims.ndim == 3:
        delta_kt = delta_sims[:, :, 0]
    else:
        delta_kt = delta_sims  # Li-Lee gives 1D deltas directly
    
    cum_kt = np.cumsum(delta_kt, axis=1)
    cum_kt = np.concatenate([np.zeros((n_sims, 1)), cum_kt], axis=1)
    
    e0_all = np.zeros((n_sims, n_steps + 1, n_countries))
    for c_idx, code in enumerate(COUNTRIES.keys()):
        log_mx_matrix = np.load(os.path.join(PROCESSED_DIR, f'{code}_log_mx_{sex}.npy'))
        log_mx_anchor = log_mx_matrix[:, anchor_year_idx]  # 2010 mortality
        for s in range(n_sims):
            for t in range(n_steps + 1):
                log_mx_t = log_mx_anchor + Bx * cum_kt[s, t]
                log_mx_t[40:91] = _IR.fit_transform(_AGES_40_90, log_mx_t[40:91])
                e0_all[s, t, c_idx] = calculate_e0(log_mx_t)
    return e0_all

print('Reconstructing e0...')
print('  AINN Male...')
ainn_e0_m = reconstruct_e0_backtest(sims_m, 'male')
print('  AINN Female...')
ainn_e0_f = reconstruct_e0_backtest(sims_f, 'female')
print('  Li-Lee Male...')
ll_e0_m = reconstruct_e0_backtest(ll_delta_m, 'male')
print('  Li-Lee Female...')
ll_e0_f = reconstruct_e0_backtest(ll_delta_f, 'female')

bt_years = list(range(CUTOFF_YEAR, CUTOFF_YEAR + N_BACKTEST + 1))  # 2010-2020
print(f'\ne0 arrays: AINN={ainn_e0_m.shape}, Li-Lee={ll_e0_m.shape}')
print(f'Backtest years: {bt_years[0]}-{bt_years[-1]}')

## 9.6: Results — Predicted vs Observed e₀

In [ ]:
# Compute MAE for each model × country × sex at 2020 (10-year horizon)
rows = []
for c_idx, (code, name) in enumerate(COUNTRIES.items()):
    for sex, ainn_e0, ll_e0 in [
        ('Male', ainn_e0_m, ll_e0_m),
        ('Female', ainn_e0_f, ll_e0_f)
    ]:
        obs_2020 = observed_e0[sex.lower()][code][-1]  # Observed e0 at 2020
        obs_2010 = observed_e0[sex.lower()][code][0]   # Observed e0 at 2010
        
        ainn_med_2020 = np.percentile(ainn_e0[:, -1, c_idx], 50)
        ll_med_2020 = np.percentile(ll_e0[:, -1, c_idx], 50)
        
        ainn_err = ainn_med_2020 - obs_2020
        ll_err = ll_med_2020 - obs_2020
        
        # Check if observed falls within 90% CI
        ainn_p5 = np.percentile(ainn_e0[:, -1, c_idx], 5)
        ainn_p95 = np.percentile(ainn_e0[:, -1, c_idx], 95)
        ll_p5 = np.percentile(ll_e0[:, -1, c_idx], 5)
        ll_p95 = np.percentile(ll_e0[:, -1, c_idx], 95)
        
        ainn_in_ci = 'YES' if ainn_p5 <= obs_2020 <= ainn_p95 else 'NO'
        ll_in_ci = 'YES' if ll_p5 <= obs_2020 <= ll_p95 else 'NO'
        
        rows.append({
            'Country': name, 'Sex': sex,
            'Obs 2010': round(obs_2010, 2),
            'Obs 2020': round(obs_2020, 2),
            'AINN Med': round(ainn_med_2020, 2),
            'LL Med': round(ll_med_2020, 2),
            'AINN Err': f'{ainn_err:+.2f}',
            'LL Err': f'{ll_err:+.2f}',
            'AINN in 90%CI': ainn_in_ci,
            'LL in 90%CI': ll_in_ci,
        })

df = pd.DataFrame(rows)
print('='*130)
print('10-YEAR BACKTEST: Predicted vs Observed e0 at 2020 (trained on 1956-2010)')
print('='*130)
print(df.to_string(index=False))

# Summary statistics
ainn_errors = [float(r['AINN Err']) for r in rows]
ll_errors = [float(r['LL Err']) for r in rows]
print(f'\nSummary (absolute error at 2020):')
print(f'  AINN — Mean: {np.mean(np.abs(ainn_errors)):.3f}, Max: {np.max(np.abs(ainn_errors)):.3f}')
print(f'  LL   — Mean: {np.mean(np.abs(ll_errors)):.3f}, Max: {np.max(np.abs(ll_errors)):.3f}')
print(f'  AINN wins: {sum(abs(a) < abs(l) for a, l in zip(ainn_errors, ll_errors))}/12 countries×sexes')

## 9.7: Visualization — CHE Backtest Trajectory

In [ ]:
che_idx = list(COUNTRIES.keys()).index('CHE')

fig, axes = plt.subplots(1, 2, figsize=(14, 6), sharey=True)

for sex_idx, (sex, ainn_e0, ll_e0) in enumerate([
    ('Male', ainn_e0_m[:, :, che_idx], ll_e0_m[:, :, che_idx]),
    ('Female', ainn_e0_f[:, :, che_idx], ll_e0_f[:, :, che_idx])
]):
    ax = axes[sex_idx]
    obs = observed_e0[sex.lower()]['CHE']
    
    # AINN fan chart
    a_p5 = np.percentile(ainn_e0, 5, axis=0)
    a_p95 = np.percentile(ainn_e0, 95, axis=0)
    a_med = np.percentile(ainn_e0, 50, axis=0)
    ax.fill_between(bt_years, a_p5, a_p95, alpha=0.15, color=COUNTRY_COLORS['CHE'])
    ax.plot(bt_years, a_med, color=COUNTRY_COLORS['CHE'], linewidth=2, label='AINN')
    
    # Li-Lee fan chart
    ll_p5 = np.percentile(ll_e0, 5, axis=0)
    ll_p95 = np.percentile(ll_e0, 95, axis=0)
    ll_med = np.percentile(ll_e0, 50, axis=0)
    ax.fill_between(bt_years, ll_p5, ll_p95, alpha=0.1, color='grey')
    ax.plot(bt_years, ll_med, color='grey', linewidth=2, linestyle='--', label='Li-Lee RWD')
    
    # Observed
    ax.plot(backtest_years, obs, color='red', linewidth=2.5, marker='o',
            markersize=4, label='Observed (HMD)', zorder=5)
    
    ax.set_xlabel('Year')
    ax.set_title(sex)
    if sex_idx == 0:
        ax.set_ylabel('$e_0$ (Switzerland)')
    ax.legend(loc='lower right', fontsize=10)

fig.suptitle('10-Year Backtest: CHE e₀ Predicted (2011-2020) vs Observed', fontsize=13, y=1.02)
plt.tight_layout()
save_dual(fig, 'fig23_backtest_10yr_che')
plt.show()

## 9.8: Conclusion

In [ ]:
print('='*80)
print('BACKTEST CONCLUSION')
print('='*80)
print()
print('The AINN was trained on 1956-2010 data only and asked to forecast')
print('life expectancy for 2011-2020 — a decade that includes the post-2011')
print('mortality deceleration and the 2020 COVID shock.')
print()
print(f'Mean absolute error at 2020 (10-year horizon):')
print(f'  AINN: {np.mean(np.abs(ainn_errors)):.3f} years')
print(f'  Li-Lee RWD: {np.mean(np.abs(ll_errors)):.3f} years')
print()
ainn_in = sum(1 for r in rows if r['AINN in 90%CI'] == 'YES')
ll_in = sum(1 for r in rows if r['LL in 90%CI'] == 'YES')
print(f'Observed e0 within 90% CI:')
print(f'  AINN: {ainn_in}/12')
print(f'  Li-Lee: {ll_in}/12')
print()
print('This backtest validates the AINN on real out-of-sample data —')
print('not on the same period used for model selection or hyperparameter tuning.')
print()

# Save results
df.to_csv(os.path.join(PROCESSED_DIR, 'backtest_10yr_results.csv'), index=False)
print(f'Results saved to {PROCESSED_DIR}backtest_10yr_results.csv')
print('\nNotebook 09 complete.')